In [11]:
!pip install scikit-learn
import pandas as pd
import numpy as np
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from xgboost import XGBRegressor


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
df = pd.read_csv("B:\Downloads\ML-Platform\data\irrigation_prediction.csv")

target = "Irrigation_Need"

X = df.drop(target, axis=1)
y = df[target]

# Identify columns
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()


<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\kusum\AppData\Local\Temp\ipykernel_11548\3561965047.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("B:\Downloads\ML-Platform\data\irrigation_prediction.csv")


In [13]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [14]:
models = {
    "linear": Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),
    
    "rf": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42))
    ]),
    
    "xgb": Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBRegressor(n_estimators=300, learning_rate=0.05))
    ])
}

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [17]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained")


linear trained
rf trained
xgb trained


In [18]:
for name, model in models.items():
    pickle.dump(model, open(f"{name}.pkl", "wb"))


In [19]:
metadata = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "target_mean": float(y.mean())
}

json.dump(metadata, open("metadata.json", "w"))

print("✅ Models saved")

✅ Models saved
